In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""

In [ ]:
%pip install unittest-xml-reporting

In [ ]:
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner
from notebookutils import mssparkutils
 
# Initialize Spark session
spark = SparkSession.builder \
    .appName("Imaging Tests") \
    .getOrCreate()
 
 
class ImagingDataValidationTests(unittest.TestCase):
 
    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, databases = []):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.databases = databases
 
    def setUp(self):
        self.token = mssparkutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
        self.runtime_context = mssparkutils.runtime.context
        print(self.runtime_context)
 
    def test_bronze_imaging_folder_is_empty(self):
 
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/Imaging/DICOM/DICOM-HDS"))
        ingest_files_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest/Imaging/DICOM/DICOM-HDS")
       
        # Assert the sample data folder exists but is empty
        self.assertEqual(0, len(ingest_files_folder))
       
    def test_bronze_lakehouse_folder_structure_is_hydrated(self):
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/External"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Failed"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Ingest"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/ReferenceData"))
        self.assertTrue(mssparkutils.fs.exists(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/SampleData"))
 
    def test_verify_resourcetype_imaging_study(self):
        expected_resource_type = "ImagingStudy"
        df = self.spark.sql("SELECT resourceType FROM healthcare1_msft_silver.ImagingStudy")
        result = df.collect()[0]['resourceType']
        self.assertEqual(expected_resource_type, result)
   
    def test_data_presence_in_imagingdicom_deltatable(self):
        df = self.spark.sql("SELECT accessionNumber, patientName, metadata_string FROM healthcare1_msft_bronze.ImagingDicom")
        row = df.collect()[0]
        accession_number, patient_name, metadata_string = row['accessionNumber'], row['patientName'], row['metadata_string']
       
        # Check if all three fields are not None
        self.assertIsNotNone(accession_number)
        self.assertIsNotNone(patient_name)
        self.assertIsNotNone(metadata_string)
       
        # Check if all three fields are not empty strings
        self.assertNotEqual(accession_number, "")
        self.assertNotEqual(patient_name, "")
        self.assertNotEqual(metadata_string, "")
 
    def test_imagingstudy_ndjson_generated_and_archived(self):
        processed_files_folder = mssparkutils.fs.ls(f"abfss://{self.workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process/Imaging/DICOM/DICOM-HDS/")
        year_folders = mssparkutils.fs.ls(processed_files_folder[0].path)
        self.assertEqual(1, len(year_folders))
 
        months_folder = mssparkutils.fs.ls(year_folders[0].path)
        self.assertEqual(1, len(months_folder))
 
        days_folder = mssparkutils.fs.ls(months_folder[0].path)
        self.assertEqual(1, len(days_folder))
 
        dicom_hds_folder = mssparkutils.fs.ls(days_folder[0].path)
        self.assertGreater(len(dicom_hds_folder), 0, "No folders found in the days_folder")
       
        dynamic_folder = dicom_hds_folder[0].path
        dynamic_folder_contents = mssparkutils.fs.ls(dynamic_folder)
       
        # Verify if dynamic_folder contains a .zip file
        zip_files = [file for file in dynamic_folder_contents if file.name.endswith('.zip')]
        self.assertGreater(len(zip_files), 0, "No .zip file found in the dynamic_folder")
 
    def test_bronze_imaging_dicom_table_is_populated(self):
        # count after unzip is 7740 for default sampledata
        #query ImagingDicom delta table
        df = self.spark.sql("SELECT patientId FROM healthcare1_msft_bronze.ImagingDicom")
        count = df.count()
        #assert count to expected
        self.assertEqual(7740, count)
   
    def test_silver_imaging_study_table_is_populated(self):
        # count after unzip is 340 for default sampledata
        #query ImagingStudy delta table
        df = self.spark.sql("SELECT * FROM healthcare1_msft_silver.ImagingStudy")
        count = df.count()
        #assert count to expected
        self.assertEqual(340, count)
   
    def test_silver_imaging_metastore_table_is_populated(self):
        # count after unzip is 7740 for default sampledata
        #query ImagingMetastore delta table
        df = self.spark.sql("SELECT * FROM healthcare1_msft_silver.ImagingMetastore")
        count = df.count()
        #assert count to expected
        self.assertEqual(7740, count)
 
 
def run_tests_and_write_output(spark):
   
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(ImagingDataValidationTests)
 
    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id
 
    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')
 
    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@msit-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output
 
report = run_tests_and_write_output(spark)